In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print("✅ Working directory set to project root:", PROJECT_ROOT)

✅ Working directory set to project root: /home/robert/Projects/Feb-Hackathon


In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
import geopandas as gpd
from shapely.geometry import shape

In [3]:
DATA_PATH = PROJECT_ROOT / "data" / "final" / "means_loneliness_quarterly.csv"
df = pd.read_csv(DATA_PATH)

In [4]:
df["quarter_label"] = df["year"].astype(str) + "_Q" + df["quarter"].astype(str)
df["geography_level"] = df["geography_level"].str.lower()

In [5]:
df["quarter_num"] = df["quarter"].astype(int)
df = df.sort_values(["year", "quarter_num"])
df.head()

,year,quarter,geography_level,geography,life_satisfaction,worthwhile,happiness,anxiety,wellbeing_index,often,some_time,occasionally,hardly_ever,never,lonely_at_least_occasionally_pct,lonely_often_or_sometimes_pct,quarter_label,quarter_num
0,2025,1,country,England,6.9,7.3,7.0,3.9,4.32,7,20,24,29,18,51,27,2025_Q1,1
1,2025,1,country,Scotland,7.0,7.4,7.0,3.7,4.42,8,15,22,33,20,45,23,2025_Q1,1
2,2025,1,country,Wales,7.1,7.3,7.0,3.8,4.40,6,15,30,28,18,51,21,2025_Q1,1
3,2025,1,national,Great Britain,6.9,7.3,7.0,3.9,4.32,7,19,24,29,19,50,26,2025_Q1,1
4,2025,1,region,East Midlands,6.8,7.2,7.0,4.0,4.25,6,21,25,26,20,52,27,2025_Q1,1


In [6]:
level_dd = widgets.Dropdown(
    options=["national", "country", "region"],
    value="national",
    description="Level:",
)

In [7]:
geo_dd = widgets.Dropdown(
    options=sorted(df.loc[df["geography_level"] ==
                   "national", "geography"].unique().tolist()),
    value="Great Britain" if "Great Britain" in df["geography"].unique(
    ) else sorted(df["geography"].unique())[0],
    description="Geography:",
)

In [8]:
quarter_dd = widgets.Dropdown(
    options=["All"] + sorted(df["quarter_label"].unique().tolist()),
    value="All",
    description="Quarter:",
)

In [9]:
metric_dd = widgets.Dropdown(
    options=[
        ("Lonely (often/sometimes) %", "lonely_often_or_sometimes_pct"),
        ("Lonely (at least occasionally) %", "lonely_at_least_occasionally_pct"),
        ("Wellbeing index", "wellbeing_index"),
        ("Life satisfaction (avg)", "life_satisfaction"),
        ("Worthwhile (avg)", "worthwhile"),
        ("Happiness (avg)", "happiness"),
        ("Anxiety (avg)", "anxiety"),
    ],
    value="lonely_often_or_sometimes_pct",
    description="Metric:",
)

In [10]:
out = widgets.Output()

In [11]:
def update_geo_options(*args):
    level = level_dd.value
    options = sorted(df.loc[df["geography_level"] ==
                     level, "geography"].unique().tolist())
    geo_dd.options = options
    # pick a sensible default
    if level == "national" and "Great Britain" in options:
        geo_dd.value = "Great Britain"
    else:
        geo_dd.value = options[0] if options else None

In [12]:
def render(*args):
    with out:
        clear_output(wait=True)

        level = level_dd.value
        geo = geo_dd.value
        quarter = quarter_dd.value
        metric = metric_dd.value

        dff = df[(df["geography_level"] == level) &
                 (df["geography"] == geo)].copy()

        if quarter != "All":
            dff = dff[dff["quarter_label"] == quarter]

        # If All: show trend across quarters; if single quarter: show bar (single point)
        if quarter == "All":
            fig = px.line(
                dff,
                x="quarter_label",
                y=metric,
                markers=True,
                title=f"{geo} — {metric_dd.label} (All quarters)",
            )
            fig.update_xaxes(categoryorder="array", categoryarray=sorted(
                df["quarter_label"].unique()))
        else:
            fig = px.bar(
                dff,
                x="quarter_label",
                y=metric,
                title=f"{geo} — {metric_dd.label} ({quarter})",
            )

        fig.update_layout(height=420, margin=dict(l=30, r=30, t=60, b=30))
        fig.show()

In [13]:
# Wire up callbacks
level_dd.observe(update_geo_options, names="value")
level_dd.observe(render, names="value")
geo_dd.observe(render, names="value")
quarter_dd.observe(render, names="value")
metric_dd.observe(render, names="value")

In [14]:
# Initial render
update_geo_options()
render()

display(widgets.HBox([level_dd, geo_dd, quarter_dd, metric_dd]))
display(out)

Output()

In [15]:
df[df["geography_level"] == "national"][[
    "quarter_label",
    "lonely_often_or_sometimes_pct",
    "wellbeing_index"
]]

,quarter_label,lonely_often_or_sometimes_pct,wellbeing_index
3,2025_Q1,26,4.32
16,2025_Q2,25,4.30
29,2025_Q3,26,4.22
42,2025_Q4,24,4.25


In [16]:
import json
import requests

In [17]:
def fetch_geojson(url: str, timeout: int = 30):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return json.loads(r.content)  # works even if content-type is octet-stream

In [18]:
# Countries (England/Scotland/Wales/Northern Ireland)
countries_geojson_url = (
    "https://open-geography-portalx-ons.hub.arcgis.com/"
    "api/download/v1/items/06d13e0421784911aa669768f25dcb18/geojson?layers=0"
)
ctry_geojson = fetch_geojson(countries_geojson_url)

eer_url = "https://raw.githubusercontent.com/martinjc/UK-GeoJSON/master/json/electoral/gb/eer.json"
eer_geojson = fetch_geojson(eer_url)

# OPTIONAL: if your dataset is Great Britain only (no Northern Ireland), remove NI so it doesn't show as blank
ctry_geojson["features"] = [
    f for f in ctry_geojson["features"]
    if f["properties"].get("CTRY22NM") != "Northern Ireland"
]

In [19]:
ctry_geojson["features"][0]["properties"].keys()


dict_keys(['FID', 'CTRY24CD', 'CTRY24NM', 'CTRY24NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID'])

In [20]:
eer_geojson["features"][0]["properties"].keys()

dict_keys(['EER13CD', 'EER13CDO', 'EER13NM'])

In [ ]:
def reproject_featurecollection_to_wgs84(featurecollection: dict, src_epsg: int = 27700) -> dict:
    """
    Convert a GeoJSON FeatureCollection from EPSG:27700 (BNG) to EPSG:4326 (lon/lat),
    returning a new GeoJSON FeatureCollection suitable for Plotly.
    """
    gdf = gpd.GeoDataFrame.from_features(
        featurecollection["features"], crs=f"EPSG:{src_epsg}")
    gdf = gdf.to_crs("EPSG:4326")

    # Keep properties + geometry as GeoJSON
    return json.loads(gdf.to_json())


# Reproject ONLY the country geojson (regions one is already lon/lat in that repo)
ctry_geojson_wgs84 = reproject_featurecollection_to_wgs84(
    ctry_geojson, src_epsg=27700)

In [ ]:
# --- Config for each map level ---
LEVELS = {
    "country": {
        "geojson": ctry_geojson_wgs84,          # <-- use reprojected
        "featureidkey": "properties.CTRY24NM",
        "title": "Countries"
    },
    "region": {
        "geojson": eer_geojson,                 # usually already OK
        "featureidkey": "properties.EER13NM",
        "title": "English Regions"
    }
}

# --- Metrics (only those present in your CSV will appear) ---
METRICS = [
    ("Lonely (often/sometimes) %", "lonely_often_or_sometimes_pct"),
    ("Lonely (at least occasionally) %", "lonely_at_least_occasionally_pct"),
    ("Wellbeing index", "wellbeing_index"),
    ("Life satisfaction (avg)", "life_satisfaction"),
    ("Worthwhile (avg)", "worthwhile"),
    ("Happiness yesterday (avg)", "happiness_yesterday"),
    ("Anxiety yesterday (avg)", "anxiety_yesterday"),
]

metric_options = [(label, col) for label, col in METRICS if col in df.columns]
if not metric_options:
    raise ValueError(
        "No expected metric columns found in df. Check df.columns.")

# --- Widgets ---
level_dd = widgets.Dropdown(
    options=[("Country", "country"), ("Region", "region")],
    value="region" if "region" in df["geography_level"].unique(
    ) else "country",
    description="Level:"
)

quarter_dd = widgets.Dropdown(
    options=sorted(df["quarter_label"].unique().tolist()),
    value=sorted(df["quarter_label"].unique().tolist())[0],
    description="Quarter:"
)

metric_dd = widgets.Dropdown(
    options=metric_options,
    value=("lonely_often_or_sometimes_pct"
           if "lonely_often_or_sometimes_pct" in df.columns
           else metric_options[0][1]),
    description="Metric:"
)

out = widgets.Output()


def render_map(*_):
    with out:
        clear_output(wait=True)

        level = level_dd.value
        quarter = quarter_dd.value
        metric = metric_dd.value
        cfg = LEVELS[level]

        dff = df[(df["geography_level"] == level) & (
            df["quarter_label"] == quarter)].copy()

        if dff.empty:
            print(f"No data for level='{level}' and quarter='{quarter}'.")
            return

        # If your CSV is Great Britain only, remove NI so it doesn't show blank on the country map
        geo = cfg["geojson"]
        if level == "country" and "Northern Ireland" not in dff["geography"].unique():
            geo = dict(cfg["geojson"])  # shallow copy
            geo["features"] = [
                f for f in cfg["geojson"]["features"]
                if f["properties"].get("CTRY24NM") != "Northern Ireland"
            ]

        fig = px.choropleth(
            dff,
            geojson=geo,
            locations="geography",
            featureidkey=cfg["featureidkey"],
            color=metric,
            hover_name="geography",
            hover_data={metric: ':.2f'},
            title=f"{cfg['title']} — {quarter} — {metric_dd.label}",
        )

        fig.update_geos(fitbounds="locations", visible=False)
        fig.update_layout(height=650, margin=dict(l=10, r=10, t=70, b=10))
        fig.show()


for w in (level_dd, quarter_dd, metric_dd):
    w.observe(render_map, names="value")

display(widgets.HBox([level_dd, quarter_dd, metric_dd]))
display(out)
render_map()

Output()